# 01 — TTS Server (XTTS-v2) on Colab

שרת FastAPI שמשרת את מודל **XTTS-v2** של Coqui, חושף HTTP endpoints דרך Cloudflare Tunnel, ושומר את הקבצים ל-Google Drive.

**חשוב:** לפני הרצה ודא שהגדרת Runtime → GPU (T4 בחינם מספיק).

## CELL 1 — התקנת ספריות

`TTS` זו ספריית Coqui שמכילה את XTTS-v2.  
`fastapi` + `uvicorn` ל-HTTP server, `nest-asyncio` כדי להריץ uvicorn בתוך Colab (שכבר מריץ event loop משלו).  
`pyngrok` שמור כ-fallback; אנחנו נשתמש ב-cloudflared.

In [ ]:
# התקנת חבילות Python הדרושות לשרת
!pip install -q TTS fastapi uvicorn nest-asyncio pyngrok
!pip install -q aiofiles httpx python-multipart

## CELL 2 — חיבור Google Drive

כל הקבצים שנוצרים (`outputs/tts`) ודגימות הקול לשיבוט (`voice_refs`) נשמרים ב-Drive כדי שיישארו אחרי שה-runtime ייסגר.

In [ ]:
# חיבור Drive — פותח חלון אימות בפעם הראשונה
from google.colab import drive
drive.mount('/content/drive')

import os

# תיקיות הפרויקט ב-Drive
OUTPUTS_DIR = '/content/drive/MyDrive/viral_empire/outputs/tts'
REFS_DIR = '/content/drive/MyDrive/viral_empire/voice_refs'

# יצירת התיקיות אם לא קיימות
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(REFS_DIR, exist_ok=True)

print(f'OUTPUTS_DIR = {OUTPUTS_DIR}')
print(f'REFS_DIR    = {REFS_DIR}')

## CELL 3 — טעינת XTTS-v2 + בדיקת דגימה

XTTS-v2 הוא מודל TTS רב-לשוני עם **voice cloning** — הוא יכול לחקות קול מתוך דגימת WAV של 6 שניות. המודל שוקל כ-1.8GB והורדה ראשונה לוקחת דקה-שתיים.

In [ ]:
# טעינת המודל ל-GPU (חובה להאיץ)
import os
import torch

# הסכמה אוטומטית לרישיון CPML של Coqui (נדרש ב-TTS>=0.22)
os.environ['COQUI_TOS_AGREED'] = '1'

from TTS.api import TTS

# בדיקת זמינות GPU
GPU_AVAILABLE = torch.cuda.is_available()
DEVICE = 'cuda' if GPU_AVAILABLE else 'cpu'
print(f'GPU available: {GPU_AVAILABLE} | device: {DEVICE}')
if GPU_AVAILABLE:
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# טעינת המודל עם טיפול ב-OOM
print('Loading XTTS-v2... (may take 1-2 min on first run)')
try:
    tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(DEVICE)
    print('Model loaded!')
except torch.cuda.OutOfMemoryError:
    # אם ה-GPU מלא — ננקה cache וננסה שוב, ואם לא — ניפול ל-CPU
    print('GPU OOM detected — clearing cache and retrying...')
    torch.cuda.empty_cache()
    try:
        tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to('cuda')
        print('Model loaded after cache clear!')
    except torch.cuda.OutOfMemoryError:
        print('Still OOM — falling back to CPU (slow but works)')
        DEVICE = 'cpu'
        tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to('cpu')

# ==========================================================
# בדיקת דגימה — מוודא שהמודל באמת עובד
# ==========================================================
import glob

SAMPLE_TEXT = 'Hello, this is a test of the XTTS voice cloning system.'
SAMPLE_OUT = '/content/_sample_test.wav'

# מחפש דגימת voice reference — אם אין, משתמש בקול ברירת-מחדל
ref_files = glob.glob(f'{REFS_DIR}/*.wav')
if ref_files:
    speaker_ref = ref_files[0]
    print(f'Using voice reference: {os.path.basename(speaker_ref)}')
    try:
        tts.tts_to_file(
            text=SAMPLE_TEXT,
            speaker_wav=speaker_ref,
            language='en',
            file_path=SAMPLE_OUT,
        )
        print(f'Sample OK: {SAMPLE_OUT} ({os.path.getsize(SAMPLE_OUT)} bytes)')
    except torch.cuda.OutOfMemoryError:
        print('GPU OOM during sample — clearing cache')
        torch.cuda.empty_cache()
else:
    print(f'(no voice reference in {REFS_DIR} — skipping sample test)')
    print('To test: upload a 6-10s clean WAV into that folder.')

## CELL 4 — אפליקציית FastAPI

שלוש נקודות קצה:
- `GET /health` — בדיקת חיים + סטטוס GPU
- `POST /tts` — מקבל טקסט + קובץ voice reference ומחזיר WAV
- `GET /jobs/{job_id}` — מבדוק אם הקובץ מוכן ב-Drive

כל התוצאות נשמרות ב-`OUTPUTS_DIR/{job_id}.wav`. אם ב-payload מועבר `callback_url`, השרת יעשה POST בחזרה עם ה-URL של הקובץ ברגע שסיים.

In [ ]:
# בניית אפליקציית FastAPI + worker thread שמריץ את uvicorn ברקע
import asyncio
import os
import threading
import traceback
from typing import Optional

import httpx
import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field

# מאפשר event loop מקונן בתוך Colab
nest_asyncio.apply()

app = FastAPI(title='Viral Empire — XTTS-v2 Server', version='0.1.0')

# ---------- סכמת בקשה ----------
class TTSRequest(BaseModel):
    job_id: str = Field(..., min_length=1)
    text: str = Field(..., min_length=1)
    language: str = Field(default='en', min_length=2, max_length=10)
    speaker_wav_filename: str = Field(
        ..., description='Filename of a WAV inside REFS_DIR (voice reference)'
    )
    callback_url: Optional[str] = None

# ---------- /health ----------
@app.get('/health')
def health():
    return {
        'status': 'ok',
        'model': 'xtts-v2',
        'gpu': bool(torch.cuda.is_available()),
        'device': DEVICE,
    }

# ---------- /jobs/{job_id} ----------
@app.get('/jobs/{job_id}')
def job_status(job_id: str):
    path = os.path.join(OUTPUTS_DIR, f'{job_id}.wav')
    if not os.path.exists(path):
        return {'job_id': job_id, 'status': 'pending', 'exists': False}
    return {
        'job_id': job_id,
        'status': 'done',
        'exists': True,
        'size_bytes': os.path.getsize(path),
        'path': path,
    }

@app.get('/jobs/{job_id}/download')
def job_download(job_id: str):
    path = os.path.join(OUTPUTS_DIR, f'{job_id}.wav')
    if not os.path.exists(path):
        raise HTTPException(404, 'job not ready')
    return FileResponse(path, media_type='audio/wav', filename=f'{job_id}.wav')

# ---------- /tts ----------
@app.post('/tts')
async def synthesize(req: TTSRequest):
    # אימות שקובץ ה-reference קיים
    ref_path = os.path.join(REFS_DIR, req.speaker_wav_filename)
    if not os.path.exists(ref_path):
        raise HTTPException(
            status_code=404,
            detail=f'speaker_wav not found: {req.speaker_wav_filename}',
        )

    out_path = os.path.join(OUTPUTS_DIR, f'{req.job_id}.wav')

    # קריאה ל-XTTS — חוסמת CPU/GPU, לכן עטפתי ב-to_thread
    def _synthesize_sync():
        tts.tts_to_file(
            text=req.text,
            speaker_wav=ref_path,
            language=req.language,
            file_path=out_path,
        )

    try:
        await asyncio.to_thread(_synthesize_sync)
    except torch.cuda.OutOfMemoryError:
        # ניקוי ה-cache של GPU וניסיון נוסף
        torch.cuda.empty_cache()
        try:
            await asyncio.to_thread(_synthesize_sync)
        except Exception as exc:
            raise HTTPException(503, f'GPU OOM after retry: {exc}') from exc
    except Exception as exc:
        traceback.print_exc()
        raise HTTPException(500, f'synthesis failed: {exc}') from exc

    result = {
        'job_id': req.job_id,
        'status': 'done',
        'path': out_path,
        'size_bytes': os.path.getsize(out_path),
        'download_url': f'/jobs/{req.job_id}/download',
    }

    # אופציונלי: webhook callback לבאקאנד המקומי
    if req.callback_url:
        try:
            async with httpx.AsyncClient(timeout=15) as http:
                await http.post(req.callback_url, json=result)
        except Exception as exc:
            # לא מפילים את הבקשה אם ה-callback נכשל
            print(f'callback failed: {exc}')
            result['callback_error'] = str(exc)

    return result

# ---------- הרצת השרת ב-thread רקע ----------
PORT = 8000

def _run_server():
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='info')

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

# מחכה שה-port יקלוט בקשות
import time, socket
for _ in range(30):
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=1):
            break
    except OSError:
        time.sleep(1)
print(f'FastAPI running on http://127.0.0.1:{PORT}')

## CELL 5 — חשיפה דרך Cloudflare Tunnel

`cloudflared` יוצר תת-דומיין HTTPS ציבורי (`*.trycloudflare.com`) בלי הרשמה ובלי תשלום. זה מה שיאפשר ל-backend המקומי לדבר עם ה-notebook.

ה-URL נשמר ל-`colab_url.txt` ב-Drive כדי שה-backend יוכל לקרוא אותו אוטומטית.

In [ ]:
# התקנת cloudflared והקמת tunnel
import os
import re
import subprocess
import time

# הורדת בינארי cloudflared (פעם אחת בלבד)
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('Installing cloudflared...')
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    print('cloudflared installed')

# הרצת tunnel ברקע עם לוג לקובץ
LOG_PATH = '/content/cloudflared.log'
if os.path.exists(LOG_PATH):
    os.remove(LOG_PATH)

proc = subprocess.Popen(
    [
        'cloudflared', 'tunnel', '--no-autoupdate',
        '--url', f'http://localhost:{PORT}',
        '--logfile', LOG_PATH,
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f'cloudflared started (pid={proc.pid}), waiting for public URL...')

# ה-URL נכתב ללוג בתוך ~15 שניות
public_url = None
url_pattern = re.compile(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com')
for _ in range(60):
    time.sleep(1)
    if not os.path.exists(LOG_PATH):
        continue
    with open(LOG_PATH, 'r') as f:
        log_text = f.read()
    m = url_pattern.search(log_text)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    raise RuntimeError(
        f'Could not find tunnel URL. Check {LOG_PATH}'
    )

# שמירה ל-Drive כך שהbackend יוכל למצוא
URL_FILE = '/content/drive/MyDrive/viral_empire/colab_url.txt'
with open(URL_FILE, 'w') as f:
    f.write(public_url + '\n')

print('=' * 60)
print(f'PUBLIC URL: {public_url}')
print(f'Saved to:   {URL_FILE}')
print('=' * 60)
print(f'Test it:    curl {public_url}/health')

## CELL 6 — לולאת Keep-Alive

Colab מנתק runtime חסר פעילות אחרי כ-90 דקות. הלולאה מדפיסה timestamp כל 30 שניות כדי שהטאב ייחשב פעיל.

**אל תסגור את הטאב הזה כל עוד אתה צריך את השרת.**

In [ ]:
# Keep-alive — לא לסגור את הטאב
import time

print('Server running. Keep this tab open.')
print(f'Public URL: {public_url}')

try:
    while True:
        time.sleep(30)
        print(f"Alive: {time.strftime('%H:%M:%S')}")
except KeyboardInterrupt:
    print('Stopped by user')